# Stage1 去重版 0bp 现金／选择滚动窗口必要上界

只读保存的 Qlib 归因窗口，报告固定路径直接分量校正的会计计数；不构造交易规则。

In [1]:
import hashlib,json,math,os,pathlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=pathlib.Path.cwd();card=root/'research/stage1_common_dedup_cash_window_bound_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
account=root/src['dedup_account_summary'];attr=root/src['dedup_attribution_card'];assert sha(account)==src['dedup_account_summary_sha256'] and sha(attr/'summary.json')==src['dedup_attribution_summary_sha256']
source=json.loads((attr/'summary.json').read_text());assert source['recorder_id']==src['rolling_Recorder_id'] and source['experiment_id']==src['rolling_experiment_id'] and source['NAS_evidence']['sha256']==src['source_NAS_sha256'] and source['NAS_scoped_restore']=='nas_scoped_restore.json'
artifact=attr/'qlib_recorder'/src['rolling_experiment_id']/src['rolling_Recorder_id']/'artifacts'/src['rolling_artifact'];assert sha(artifact)==src['rolling_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
with R.uri_context(str(attr/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['rolling_experiment_id'],recorder_id=src['rolling_Recorder_id']);assert rec.status=='FINISHED';saved=rec.load_object(src['rolling_artifact'])
win=saved['C_common_dedup'];assert len(win)==819 and win.first_trade.is_unique
fields=['cash_opportunity_log','SOXL_selection_log','direct_fees_log','timing_and_benchmark_residual_log'];assert np.max(np.abs(win[fields].sum(axis=1)-win.relative_log_excess))<1e-11
actual=win.relative_log_excess.to_numpy(float);cash=win.cash_opportunity_log.to_numpy(float);selection=win.SOXL_selection_log.to_numpy(float);positive=actual>0;need=math.ceil(.70*len(win))
repaired_cash=actual-np.minimum(cash,0);repaired_selection=actual-np.minimum(selection,0);repaired_both=repaired_cash-np.minimum(selection,0);ranked=np.sort(actual)[::-1]
result={'decision':'read_only_dedup_0bp_fixed_path_cash_vs_selection_necessary_window_bound_no_policy','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['rolling_Recorder_id'],'windows':len(win),'required_positive_windows':need,'original_positive_windows':int(positive.sum()),'additional_positive_windows_required':need-int(positive.sum()),'nonwinning_with_negative_cash':int(((~positive)&(cash<0)).sum()),'nonwinning_with_negative_selection':int(((~positive)&(selection<0)).sum()),'perfect_direct_component_removal_positive_windows':{'negative_cash_only':int((repaired_cash>0).sum()),'negative_selection_only':int((repaired_selection>0).sum()),'both_negative_components':int((repaired_both>0).sum())},'cash_only_eligible_failure_windows':int(((~positive)&(cash<0)).sum()),'cash_only_minimum_fraction_of_eligible_failures_to_flip':(need-int(positive.sum()))/int(((~positive)&(cash<0)).sum()),'uniform_126_window_log_lift_infimum_for_70pct':float(max(0,-ranked[need-1])),'window_overlap':True,'direct_repair_is_native_account':False,'causal_policy_inference':False,'market_requests':0,'model_fits':0,'native_accounts':0,'bootstraps':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "decision": "read_only_dedup_0bp_fixed_path_cash_vs_selection_necessary_window_bound_no_policy",
  "plan_sha256": "5fb8de28ca3a7a3b95ade01cf49eebf956be2fc8762d8979f3bc01c4a07621ae",
  "source_recorder_id": "a10c938967a2433cbf80de74d924f4c1",
  "windows": 819,
  "required_positive_windows": 574,
  "original_positive_windows": 369,
  "additional_positive_windows_required": 205,
  "nonwinning_with_negative_cash": 418,
  "nonwinning_with_negative_selection": 158,
  "perfect_direct_component_removal_positive_windows": {
    "negative_cash_only": 659,
    "negative_selection_only": 399,
    "both_negative_components": 689
  },
  "cash_only_eligible_failure_windows": 418,
  "cash_only_minimum_fraction_of_eligible_failures_to_flip": 0.4904306220095694,
  "uniform_126_window_log_lift_infimum_for_70pct": 0.03861590034435941,
  "window_overlap": true,
  "direct_repair_is_native_account": false,
  "causal_policy_inference": false,
  "market_requests": 0,
  "model_fits": 0,
  "native_accounts":

## 中文结论

去重版原路径369/819正窗口，574要求还差205。保存分量负现金直接归零为659正窗，负SOXL选择归零仅399，两者归零689。现金为负的失败窗418个，至少205个需转正。重叠窗口及账户反馈意味着这些都不是可交易策略或原生账户，0拟合0账户，不晋级。